# ML-10 — Content Action Playbook

This notebook constructs a human-reviewed **Content Action Playbook** for **Lane 2 — Refresh / Content Opportunity Scoring**.
We translate our validated machine learning model probabilities and baseline rule scores into structured, actionable editorial queues with transparent reason codes, archetype-to-action mappings, explicit human-review boundaries, a strict **No-Go list**, and monitoring/retrain triggers.

> **Skills Loaded:** `writing-honest-claims` + `flyrank-data`

## 1. Ranked actions + reason codes

### Archetype to Action Mapping
Rather than outputting raw floating-point probabilities, the playbook maps every content item into five decision archetypes:

| Archetype | Condition / Reason Code | Primary Action Label | Editorial Objective |
|---|---|---|---|
| **Stale High-Demand Page** | `high_model_risk_stale` (Model Prob $\ge 0.70$, Days Since Update $\ge 90$) | `refresh_content` | Update outdated facts, statistics, and subheadings |
| **Striking Distance Decay** | `striking_distance_decay` (Model Prob $\ge 0.60$, Avg Position 10–25) | `refresh_and_expand` | Add missing subtopics to push page onto Page 1 |
| **Underperforming Title/CTR** | `ctr_underperformance` (Model Prob $\ge 0.60$, Position $\le 20$, CTR $< 0.5\%$) | `optimize_title_ctr` | Rewrite title tag and meta description for higher CTR |
| **Thin Visible Content** | `thin_content_gap` (Word Count $< 1,200$, Impressions $\ge 250$) | `expand_depth` | Expand article depth to address search intent |
| **Stable / Low Risk** | `routine_monitoring` (All other pages) | `monitor` | Maintain standard monitoring; no immediate edit needed |

Below, we compute the blended playbook score ($0.60 \cdot \text{Model Prob} + 0.40 \cdot \text{Baseline Score}$) and rank the queue.

In [1]:
import os
import json
import pandas as pd
import numpy as np

# Load starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
if 'is_declining_label' not in df.columns:
    df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Load Week-4 baseline & Week-5 model probabilities
baseline_path = '../outputs/baseline_action_score.csv'
model_pred_path = '../outputs/model_predictions.csv'

if os.path.exists(baseline_path):
    b_df = pd.read_csv(baseline_path)
    df = df.merge(b_df[['content_id', 'baseline_action_score']], on='content_id', how='left')
else:
    vis = df['impressions_90d'].rank(pct=True)
    fresh = df['days_since_last_update'].rank(pct=True)
    pos = df['avg_position']
    pos_risk = np.where((pos > 0) & (pos <= 20), 1.0 - (pos / 25.0), 0.2)
    ctr_gap = (1.0 - df['ctr'].rank(pct=True)) * (df['impressions_90d'] >= 100).astype(int)
    df['baseline_action_score'] = (0.40 * vis + 0.30 * fresh + 0.20 * pos_risk + 0.10 * ctr_gap).clip(0, 1)

if os.path.exists(model_pred_path):
    m_df = pd.read_csv(model_pred_path)
    df = df.merge(m_df[['content_id', 'prob_gradient_boosting_n_100']], on='content_id', how='left')
    df['gb_prob'] = df['prob_gradient_boosting_n_100'].fillna(0)
else:
    df['gb_prob'] = df['baseline_action_score']

# Composite Playbook Score
df['playbook_score'] = 0.60 * df['gb_prob'] + 0.40 * df['baseline_action_score']

# Rule mapping
def assign_playbook_action(row):
    prob = row['gb_prob']
    stale = row['days_since_last_update'] >= 90
    pos = row['avg_position']
    thin = row['word_count'] > 0 and row['word_count'] < 1200
    low_ctr = row['ctr'] < 0.5 and pos > 0 and pos <= 20
    
    if prob >= 0.70 and stale:
        return 'refresh_content'
    elif prob >= 0.60 and low_ctr:
        return 'optimize_title_ctr'
    elif prob >= 0.60 and pos > 0 and pos <= 20:
        return 'refresh_and_expand'
    elif thin and row['impressions_90d'] >= 250:
        return 'expand_depth'
    else:
        return 'monitor'

def assign_reason_code(row):
    prob = row['gb_prob']
    stale = row['days_since_last_update'] >= 90
    pos = row['avg_position']
    thin = row['word_count'] > 0 and row['word_count'] < 1200
    low_ctr = row['ctr'] < 0.5 and pos > 0 and pos <= 20
    
    if prob >= 0.70 and stale:
        return 'high_model_risk_stale'
    elif prob >= 0.60 and low_ctr:
        return 'ctr_underperformance'
    elif prob >= 0.60 and pos > 0 and pos <= 20:
        return 'striking_distance_decay'
    elif thin and row['impressions_90d'] >= 250:
        return 'thin_content_gap'
    else:
        return 'routine_monitoring'

def assign_human_review(row):
    if row['impressions_90d'] >= 50000:
        return 'MANDATORY: High traffic volume asset'
    elif 0 < row['avg_position'] <= 3.0:
        return 'MANDATORY: Top-3 ranking URL'
    elif 0.45 <= row['gb_prob'] <= 0.55:
        return 'RECOMMENDED: Boundary prediction'
    elif row['action_label'] != 'monitor':
        return 'ROUTINE: Pre-update verification'
    else:
        return 'NO_REVIEW: Automated monitoring'

df['action_label'] = df.apply(assign_playbook_action, axis=1)
df['primary_reason_code'] = df.apply(assign_reason_code, axis=1)
df['human_review_rule'] = df.apply(assign_human_review, axis=1)

df['playbook_rank'] = df['playbook_score'].rank(method='first', ascending=False).astype(int)
df_sorted = df.sort_values('playbook_rank')

print("Top 10 Actionable Queue Recommendations:")
actionable = df_sorted[df_sorted['action_label'] != 'monitor'].head(10)
print(actionable[['playbook_rank', 'content_id', 'client_id', 'playbook_score', 'gb_prob', 'action_label', 'primary_reason_code', 'human_review_rule', 'impressions_90d', 'avg_position', 'days_since_last_update']].to_string())

Top 10 Actionable Queue Recommendations:
       playbook_rank            content_id          client_id  playbook_score   gb_prob     action_label    primary_reason_code                 human_review_rule  impressions_90d  avg_position  days_since_last_update
4076               1  content_66458ac1b739  client_8527a891e2        0.865569  0.926426  refresh_content  high_model_risk_stale      MANDATORY: Top-3 ranking URL             6822           2.9                     102
13215              2  content_98aa0aecb1d9  client_8527a891e2        0.804560  0.818469  refresh_content  high_model_risk_stale  ROUTINE: Pre-update verification             5190           5.1                     104
7367               3  content_32b4e5a2f630  client_8527a891e2        0.796669  0.869871  refresh_content  high_model_risk_stale  ROUTINE: Pre-update verification             2663           6.5                     102
10136              4  content_df71843dcd17  client_8527a891e2        0.784877  0.834579  re

## 2. Intended use and limits

### Intended Operational Scope
* **Target Persona:** Content Editors, SEO Managers, and Copywriters.
* **Use Case:** Weekly editorial queue prioritization. Editors take the top 20–50 recommendations from the playbook queue each week to perform manual reviews, update statistics, expand thin sections, or adjust underperforming title tags.

### Explicit Boundary & Limitations
1. **Non-Real-Time:** Built on trailing 90-day search aggregations. Not suitable for intraday rank tracking or real-time indexing emergencies.
2. **Content-Age Window:** Valid only for established articles (≥ 90 days old). Newly published URLs (<90 days old) are excluded as they undergo natural search rank discovery.
3. **Non-Production Warning:** This model is a research decision-support tool. It is not an autonomous CMS web-hook agent and must never auto-overwrite live production URLs without human verification.

## 3. Human review + the no-go list

### Mandatory Human Review Criteria
Before an editor executes any recommended action, they must complete a 4-point verification check:
1. **Search Intent Alignment:** Confirm the target query intent hasn't shifted from informational to commercial/transactional.
2. **Information Accuracy:** Verify that statistics, dates, product prices, or screenshots are genuinely out of date.
3. **SERP Layout Shift:** Check whether rank drops stem from new Google SERP features (e.g. AI Overviews, ads, video carousels) rather than content staleness.
4. **Brand Tone Verification:** Ensure suggested content additions match the client's brand voice.

### The Strict NO-GO List (What Should NEVER Be Automated)
* 🚫 **Automated CMS Overwrites:** Direct LLM auto-publishing to live production URLs without human review is strictly prohibited due to hallucination and brand safety risks.
* 🚫 **Pillar / Top-3 Ranking URLs:** Articles occupying Top 3 search positions (`avg_position` ≤ 3.0) generate core organic revenue. Automated edits risk de-indexing winning keywords.
* 🚫 **High-Traffic Assets ($>50,000$ impressions):** High-volume URLs require multi-stakeholder approval prior to rewriting.
* 🚫 **Seasonal & Event Queries:** Temporary traffic drops driven by holiday or seasonal query cycles must not trigger permanent content rewrites.

### Cost / Value Economics
* **Review Cost:** ~0.5 hours of editor time (~$25 per URL).
* **Intervention Value:** Preventing traffic decay on a 10,000 impression URL recovers hundreds of organic clicks.
* **Efficiency Gain:** Prioritizing the top 50 high-confidence recommendations achieves **84.0% Precision@50** (vs 52.5% base rate), maximizing ROI per editor hour.

In [2]:
# Print No-Go Rule Distribution across current inventory
nogo_summary = pd.Series({
    'Top-3 Ranking URLs (Protected)': (df['avg_position'] > 0) & (df['avg_position'] <= 3.0),
    'High-Traffic Assets (>50k imps)': df['impressions_90d'] >= 50000,
    'Boundary Predictions (0.45 - 0.55 prob)': (df['gb_prob'] >= 0.45) & (df['gb_prob'] <= 0.55),
}).apply(lambda s: s.sum())

print("--- NO-GO & MANDATORY HUMAN REVIEW SUMMARY ---")
print(nogo_summary.to_string())

--- NO-GO & MANDATORY HUMAN REVIEW SUMMARY ---
Top-3 Ranking URLs (Protected)             1141
High-Traffic Assets (>50k imps)             520
Boundary Predictions (0.45 - 0.55 prob)     488


## 4. Monitoring / retrain triggers

To prevent recommendation staleness, we define four operational triggers that mandate model retraining or pipeline maintenance:

1. **Precision@50 Drift Trigger:** If the observed Precision@50 on weekly human editor reviews drops below **70.0%** (base rate 52.5%), trigger model retrain.
2. **Google Core Update Trigger:** Mandatory model re-evaluation and feature vector retrain 30 days after any major Google core algorithm update.
3. **Feature Distribution Drift:** If mean `days_since_last_update` or position distributions across client portfolios shift by $>15\%$, re-scale normalization parameters.
4. **Data Pipeline Freshness:** If the Hugging Face warehouse snapshot date slips by $>14$ days, alert data engineering.

## 5. Exports for the paper

We export the final Action Playbook queue CSVs and generate reusable publication-ready figures in both `work/figures/` and `work/outputs/`.

In [3]:
import matplotlib.pyplot as plt

# Directories
fig_dir = '../figures'
out_dir = '../outputs'
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(out_dir, exist_ok=True)

# Export Queue CSVs
queue_cols = [
    'playbook_rank', 'content_id', 'client_id', 'playbook_score', 'gb_prob', 
    'baseline_action_score', 'action_label', 'primary_reason_code', 
    'human_review_rule', 'impressions_90d', 'days_since_last_update', 
    'avg_position', 'word_count', 'ctr', 'is_declining_label'
]
df_export = df_sorted[queue_cols]
df_export.to_csv(os.path.join(out_dir, 'action_playbook_queue.csv'), index=False)
df_export.to_csv(os.path.join(out_dir, 'refresh_queue.csv'), index=False)
print(f"Saved action_playbook_queue.csv and refresh_queue.csv (Shape: {df_export.shape})")

# Save Metrics Receipt
playbook_metrics = {
    'total_inventory_pages': len(df),
    'actionable_pages_count': int((df['action_label'] != 'monitor').sum()),
    'action_mix_breakdown': df['action_label'].value_counts().to_dict(),
    'human_review_breakdown': df['human_review_rule'].value_counts().to_dict(),
    'precision_at_50_heldout_clients': 0.8400,
    'base_rate_heldout_clients': 0.5250
}
with open(os.path.join(out_dir, 'playbook_metrics.json'), 'w') as f:
    json.dump(playbook_metrics, f, indent=2)
print("Saved playbook_metrics.json receipt")

# Figure 1: Action Mix Distribution
plt.figure(figsize=(8, 4.5))
action_counts = df_sorted['action_label'].value_counts()
colors = ['#2b5c8f', '#d95f02', '#7570b3', '#e7298a', '#66a61e']
plt.bar(action_counts.index, action_counts.values, color=colors[:len(action_counts)])
plt.title('Content Action Playbook — Recommended Action Mix', fontsize=12, fontweight='bold')
plt.xlabel('Recommended Action Label', fontsize=10)
plt.ylabel('Content Item Count (Pages)', fontsize=10)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'action_mix.png'), dpi=150)
plt.savefig(os.path.join(out_dir, 'action_mix.png'), dpi=150)
plt.close()

# Figure 2: Model vs Baseline Precision@K Comparison
plt.figure(figsize=(8, 4.5))
models_name = ['Baseline (Rule)', 'Logistic Reg', 'Decision Tree', 'Random Forest', 'Gradient Boosting']
p50_scores = [0.4400, 0.6800, 0.6600, 0.3600, 0.8400]
base_rate_val = 0.5250

x = np.arange(len(models_name))
bars = plt.bar(x, p50_scores, color=['#7f7f7f', '#aec7e8', '#1f77b4', '#aec7e8', '#2ca02c'])
plt.axhline(y=base_rate_val, color='r', linestyle='--', label=f'Base Rate ({base_rate_val:.3f})')
plt.title('Model vs Baseline — Precision@50 on Held-Out Client Test Set', fontsize=12, fontweight='bold')
plt.xlabel('Model / Baseline Strategy', fontsize=10)
plt.ylabel('Precision@50 Score', fontsize=10)
plt.xticks(x, models_name, rotation=15, ha='right')
plt.ylim(0, 1.0)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.2f}', ha='center', va='bottom', fontsize=9)
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'model_vs_baseline.png'), dpi=150)
plt.savefig(os.path.join(out_dir, 'model_vs_baseline.png'), dpi=150)
plt.close()

print("Successfully generated and saved action_mix.png and model_vs_baseline.png to work/figures/ and work/outputs/")

Saved action_playbook_queue.csv and refresh_queue.csv (Shape: (30000, 15))
Saved playbook_metrics.json receipt
Successfully generated and saved action_mix.png and model_vs_baseline.png to work/figures/ and work/outputs/


## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.